# What does a neural network see? Classifying images with AlexNet
**Background:** Krizhevsky, A., Sutskever, I., & Hinton, G. E. (2012). ImageNet classification with deep convolutional neural networks. *NeurIPS, 25*.

AlexNet was trained on the **ImageNet** dataset: about 1.2 million photos, each labelled with one of **1,000 categories**. Today we test it on three kinds of images:

| Set | What it is | Question |
|---|---|---|
| 1 | ImageNet images | Can it recognise the kind of image it was trained on? |
| 2 | **Same categories**, but not from ImageNet (your own photos, drawings, unusual views) | Does it still recognise them when they look different? |
| 3 | **Categories AlexNet does not know** | What does it do when the right answer is not on its list? |

**What you will do**
1. Load AlexNet and look at the 1,000 labels it knows
2. See how an image is prepared for the network
3. Classify the three image sets and look at the top-5 predictions
4. Compare how confident AlexNet is on each set

## Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import torch
from torchvision import models, transforms
device = "cpu"

In [ ]:
# Find the course data in Sciebo (works even if the shared folder was renamed)
matches = list(Path.home().glob("sciebo/*/Week3/"))
if not matches:
    raise FileNotFoundError(
        "Course data not found. Did you connect Sciebo when starting the server, "
        "and accept the shared CNU folder in Sciebo?"
    )
DATA = matches[0]

# One folder per image set
IMAGENET_DIR      = DATA / "1_imagenet"        # images from ImageNet
SAME_CATEGORY_DIR = DATA / "2_same_category"   # same categories, but not from ImageNet
NEW_CATEGORY_DIR  = DATA / "3_new_category"    # categories AlexNet was not trained on
print(f"Using data from {DATA}")

## 1. Load AlexNet
The **weights** are the numbers the network learned during training on ImageNet. Without them, AlexNet is just an empty architecture that guesses randomly.

In [ ]:
# Load AlexNet with its trained weights (like ResMem(pretrained=True)).
# The first time, the weights (about 230 MB) are downloaded from the internet;
# after that they are stored on your server and load quickly.
weights = models.AlexNet_Weights.IMAGENET1K_V1   # the official ImageNet-trained version
model = models.alexnet(weights=weights)

# .eval() switches off training-only behaviour (dropout), so predictions are stable
model = model.to(device).eval()

# The names of the 1,000 categories, in the order of the network's outputs
categories = weights.meta["categories"]
print(f"AlexNet knows {len(categories)} categories")

In [ ]:
# What does the network look like? Note the 5 convolutional layers ("features")
# and the 3 fully connected layers ("classifier"). The last layer has 1,000 outputs:
# one for each category.
print(model)

### Which categories does AlexNet know?
AlexNet can only answer with one of these 1,000 labels. Let's look at some of them.

In [ ]:
# 20 random labels
rng = np.random.default_rng(0)        # change the number to see other labels
for i in rng.choice(len(categories), 20, replace=False):
    print(i, categories[i])

In [ ]:
def find_labels(word):
    """Print all labels that contain `word`."""
    hits = [(i, c) for i, c in enumerate(categories) if word.lower() in c.lower()]
    for i, c in hits:
        print(i, c)
    print(f"--> {len(hits)} labels contain '{word}'")

find_labels("retriever")

**Try it:** search for `"cat"`, `"car"`, `"dog"`, `"person"`, `"chair"`.

- Labels 151–268 are all **dog breeds**: 118 of the 1,000 categories! Why might that be a problem if you want to compare AlexNet with how people name objects?
- Is there a label for "person"? What does that mean for photos of people?
- Before section 5: check whether the objects in Set 3 really are missing from the list.

## 2. How an image is prepared for the network
AlexNet always takes a 224 × 224 pixel image. Every image is **resized**, **cropped** to the centre, and its colours are **normalised** the same way as during training.

In [ ]:
# The full preparation that goes into the network
preprocess = transforms.Compose([
    transforms.Resize(256),        # shorter side -> 256 pixels
    transforms.CenterCrop(224),    # cut out the central 224 x 224 square
    transforms.ToTensor(),         # turn the image into numbers (PyTorch's tensor format)
    transforms.Normalize(mean=[0.485, 0.456, 0.406],   # same colour normalisation
                         std=[0.229, 0.224, 0.225]),   # as in ImageNet training
])

# The same steps without the number conversion, so we can SHOW what the network sees
what_network_sees = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])

In [ ]:
def load_folder(folder):
    """Load all images in `folder`. The filename (without extension) is used as the image's name,
    so call your files e.g. banana.jpg or golden_retriever_drawing.png."""
    extensions = {".jpg", ".jpeg", ".png", ".webp"}
    paths = sorted(p for p in folder.iterdir() if p.suffix.lower() in extensions)
    images = [Image.open(p).convert("RGB") for p in paths]   # RGB = 3 colour channels
    names = [p.stem.replace("_", " ") for p in paths]
    print(f"{len(images)} images loaded from {folder.name}")
    return images, names

imagenet_images, imagenet_names = load_folder(IMAGENET_DIR)
same_images, same_names         = load_folder(SAME_CATEGORY_DIR)
new_images, new_names           = load_folder(NEW_CATEGORY_DIR)

In [ ]:
# Original image vs. what the network actually sees
img = same_images[0]          # try other images: any list, any index
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img); axes[0].set_title(f"original {img.size[0]} x {img.size[1]}")
axes[1].imshow(what_network_sees(img)); axes[1].set_title("network input 224 x 224")
for ax in axes: ax.axis("off")
plt.show()

Is anything important cut off by the crop? Keep this in mind when the network makes mistakes.

## 3. Helper functions: classify and show the top-5

In [ ]:
def classify(imgs, k=5):
    """Return the k most likely labels and their probabilities for each image."""
    # Prepare all images and stack them into one batch
    batch = torch.stack([preprocess(img) for img in imgs]).to(device)

    # We only predict, we don't train: no gradients needed
    with torch.no_grad():
        outputs = model(batch)                    # shape: (number of images, 1000)

    # Softmax turns the 1,000 raw outputs into probabilities that sum to 1
    probs = torch.softmax(outputs, dim=1)

    # Keep the k highest probabilities and their category numbers
    top_probs, top_idx = probs.topk(k, dim=1)

    results = []
    for p_row, i_row in zip(top_probs, top_idx):
        results.append([(categories[i], p.item()) for p, i in zip(p_row, i_row)])
    return results


def show_images(images, names=None, n_cols=5, title=None):
    """Show images in a grid, with their names underneath (if given)."""
    n_rows = int(np.ceil(len(images) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.6 * n_cols, 2.8 * n_rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for i, img in enumerate(images):
        axes[i].imshow(what_network_sees(img))
        if names is not None:
            axes[i].set_title(names[i], fontsize=9)
    if title:
        fig.suptitle(title)
    plt.tight_layout(); plt.show()


def show_top5(images, names, results, title=None):
    """One row per image: the image on the left, its top-5 predictions on the right."""
    n = len(images)
    fig, axes = plt.subplots(n, 2, figsize=(9, 2.3 * n), gridspec_kw={"width_ratios": [1, 1.7]})
    axes = np.atleast_2d(axes)
    for row, (img, name, res) in enumerate(zip(images, names, results)):
        ax_img, ax_bar = axes[row]
        ax_img.imshow(what_network_sees(img)); ax_img.axis("off")
        ax_img.set_title(name, fontsize=10)

        labels = [label for label, _ in res][::-1]   # reversed, so the top guess is on top
        probs  = [p for _, p in res][::-1]
        ax_bar.barh(labels, probs, color="tab:blue")
        ax_bar.set_xlim(0, 1)
        for y, p in enumerate(probs):
            ax_bar.text(p + 0.01, y, f"{p:.2f}", va="center", fontsize=9)
    axes[-1, 1].set_xlabel("probability")
    if title:
        fig.suptitle(title)
    plt.tight_layout(); plt.show()

## 4. Set 1: ImageNet images
### Your turn first
**Before running the next cell**, look at the images and write down what you think AlexNet will answer. Be as specific as you think the network will be!

In [ ]:
show_images(imagenet_images, title="Set 1: ImageNet images. What will AlexNet say?")

In [ ]:
imagenet_results = classify(imagenet_images)
show_top5(imagenet_images, imagenet_names, imagenet_results, title="Set 1: ImageNet images")

**Think about it:**
- How many did AlexNet get right on the first guess? In the top 5?
- Were your guesses as specific as AlexNet's labels?
- Look at the 2nd–5th guesses. Do they **look** similar to the right answer, or do they **mean** something similar? Or both?

## 5. Set 2: Same categories, but not from ImageNet
These images show categories AlexNet knows, but they look different from typical ImageNet photos: your own photos, drawings, toys, unusual viewpoints.

In [ ]:
show_images(same_images, same_names, title="Set 2: same categories, new images. Will AlexNet still recognise them?")

In [ ]:
same_results = classify(same_images)
show_top5(same_images, same_names, same_results, title="Set 2: same categories, not from ImageNet")

**Think about it:**
- Which images did AlexNet still get right? Which ones broke it?
- What is different about the ones it got wrong: the style (drawing vs. photo), the viewpoint, the background, the colour?
- Would a 5-year-old recognise these images? (Keep this question for Week 7!)

## 6. Set 3: Categories AlexNet does not know
None of these objects has a label among the 1,000 categories. But AlexNet **has to** answer: its probabilities always add up to 1. There is no "I don't know" option.

In [ ]:
show_images(new_images, new_names, title="Set 3: unknown categories. What will AlexNet say?")

In [ ]:
new_results = classify(new_images)
show_top5(new_images, new_names, new_results, title="Set 3: categories AlexNet does not know")

**Think about it:**
- Do the guesses make sense to you? Are they **visually** related (shape, colour, texture) or **semantically** related (same kind of thing, found in the same place)?
- How confident is AlexNet on these images compared with Set 1?

## 7. Compare the three sets: how confident is AlexNet?
For each image we take the probability of AlexNet's **first guess**. Is the network less sure when the image is unusual, or when it doesn't know the category at all?

In [ ]:
sets = {
    "1: ImageNet":          imagenet_results,
    "2: same category":     same_results,
    "3: unknown category":  new_results,
}

plt.figure(figsize=(6, 4))
for x, (set_name, results) in enumerate(sets.items()):
    top1 = [res[0][1] for res in results]                 # probability of the first guess
    jitter = np.random.default_rng(x).uniform(-0.1, 0.1, len(top1))
    plt.scatter(x + jitter, top1, alpha=0.6)              # one dot per image
    plt.plot([x - 0.25, x + 0.25], [np.mean(top1)] * 2, color="black")   # mean
    print(f"{set_name:22s} mean top-1 probability = {np.mean(top1):.2f}")
plt.xticks(range(len(sets)), sets.keys())
plt.ylabel("probability of first guess")
plt.ylim(0, 1)
plt.title("How confident is AlexNet?")
plt.show()

**Think about it:**
- Is AlexNet less confident on Set 3? Is it ever **very** confident and wrong?
- Can we use the confidence to tell whether the network "knows" what it is looking at?

## 8. Explore: bring your own image
Upload a photo (drag it into the file browser on the left), put its filename below, and run the cell. Can you find an image that AlexNet classifies with high confidence but completely wrong?

In [ ]:
my_file = "my_image.jpg"     # <- your filename here
my_image = Image.open(my_file).convert("RGB")
show_top5([my_image], [my_file], classify([my_image]), title="Your image")

## 9. Discussion questions
1. AlexNet is very good at Set 1. Does that make it a good model of how **humans** recognise objects? What else would you want to test?
2. ImageNet has 118 dog breeds but no "person". How does the choice of training labels shape what a network "sees"?
3. Where AlexNet's wrong guesses were sensible, were they sensible in the way a human's mistakes would be? (This is the question of Week 6.)